# Agente de CVs: RAG por payload + Qdrant (vectorial gratis) + Hugging Face
Basado en la idea del repo *azure-ai-search-embeddings*: indexar **título + keywords + embedding** y no dejar que frases como "Hoja de vida" dominen la búsqueda.
Aquí el equivalente gratuito: **Qdrant** (payload con habilidades/años/idiomas + vector) y embeddings/LLM de **Hugging Face**.

| Azure AI Search (repo) | Este notebook |
|---|---|
| campo `title` + `keywords` | `candidato` + payload `habilidades` |
| campo `embedding` (HNSW) | vector en Qdrant (HNSW, coseno) |
| búsqueda híbrida + semantic ranker | filtro por payload + vector + re-puntuación por diccionario |

In [1]:
!pip install -q huggingface_hub numpy pypdf qdrant-client

In [ ]:
import os, re, json, getpass, unicodedata, uuid
import numpy as np
from huggingface_hub import InferenceClient
from qdrant_client import QdrantClient, models

HF_TOKEN = os.getenv("HF_TOKEN") or getpass.getpass("HF token: ")   # https://huggingface.co/settings/tokens
EMB_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
LLM_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
client = InferenceClient(token=HF_TOKEN)

# ---- Qdrant: elige UNA opción ----
QDRANT_URL = os.getenv("QDRANT_URL")          # Opción B: Qdrant Cloud gratis (https://cloud.qdrant.io)
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
COLECCION = "cvs"
qd = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY) if QDRANT_URL else QdrantClient(path="./qdrant_data")  # Opción A: local

## 1. Núcleo (payload + Qdrant + agente)

In [ ]:
CATALOGO = ["python", "sql", "spark", "airflow", "aws", "azure", "gcp", "docker", "kubernetes",
            "fastapi", "django", "flask", "postgresql", "mongodb", "pytorch", "tensorflow",
            "nlp", "hugging face", "rag", "llm", "faiss", "langchain", "machine learning",
            "pandas", "etl", "java", "javascript", "react", "git", "figma", "photoshop"]
ALIAS = {"huggingface": "hugging face", "hf": "hugging face", "ml": "machine learning",
         "postgres": "postgresql", "llms": "llm", "js": "javascript", "k8s": "kubernetes"}
IDIOMAS = ["ingles", "espanol", "frances", "portugues", "aleman"]


def norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s).lower())
    return "".join(c for c in s if not unicodedata.combining(c)).strip()


def canon(t: str) -> str:
    t = norm(t)
    return ALIAS.get(t, t)


def _palabra(term: str, texto: str) -> bool:
    return re.search(rf"(?<![a-z0-9]){re.escape(term)}(?![a-z0-9])", texto) is not None


def payload_por_reglas(texto: str) -> dict:
    """Extracción rápida sin LLM: catálogo + regex."""
    t = norm(texto)
    hab = {s for s in CATALOGO if _palabra(s, t)}
    hab |= {c for a, c in ALIAS.items() if _palabra(a, t)}
    anios = [int(x) for x in re.findall(r"(\d+)\s*\+?\s*anos", t)]
    return {"habilidades": sorted(hab), "anios": max(anios) if anios else None,
            "idiomas": [i for i in IDIOMAS if i in t]}


def llm(system: str, user: str, max_tokens: int = 600) -> str:
    r = client.chat_completion(model=LLM_MODEL, max_tokens=max_tokens, temperature=0.1,
                               messages=[{"role": "system", "content": system},
                                         {"role": "user", "content": user}])
    return r.choices[0].message.content


def _json(txt: str) -> dict:
    try:
        return json.loads(re.search(r"\{.*\}", txt, re.S).group())
    except Exception:
        return {}


def payload_cv(texto: str) -> dict:
    """Payload del CV = LLM (estructura) + reglas (red de seguridad)."""
    base = payload_por_reglas(texto)
    d = _json(llm('Extrae del CV SOLO JSON: {"habilidades":[...],"anios":int|null,"idiomas":[...]}. '
                  "Habilidades = tecnologías y herramientas, en minúscula.", texto[:3500], 300))
    hab = {canon(h) for h in d.get("habilidades", []) if isinstance(h, str)} | set(base["habilidades"])
    anios = d.get("anios") if isinstance(d.get("anios"), int) else base["anios"]
    idi = {norm(i) for i in d.get("idiomas", []) if isinstance(i, str)} | set(base["idiomas"])
    return {"habilidades": sorted(hab), "anios": anios, "idiomas": sorted(idi)}


def payload_pregunta(pregunta: str) -> dict:
    """Convierte la pregunta en diccionario de términos clave."""
    d = _json(llm('Convierte la pregunta de un reclutador en SOLO JSON: '
                  '{"habilidades":[...],"anios_min":int|null,"idiomas":[...]}. '
                  "Solo términos clave, en minúscula, sin frases.", pregunta, 200))
    base = payload_por_reglas(pregunta)
    hab = {canon(h) for h in d.get("habilidades", []) if isinstance(h, str)} | set(base["habilidades"])
    anios = d.get("anios_min") if isinstance(d.get("anios_min"), int) else base["anios"]
    idi = {norm(i) for i in d.get("idiomas", []) if isinstance(i, str)} | set(base["idiomas"])
    return {"habilidades": sorted(hab), "anios_min": anios, "idiomas": sorted(idi)}


def embed(texto: str) -> np.ndarray:
    v = np.array(client.feature_extraction(texto, model=EMB_MODEL), dtype=np.float32)
    if v.ndim > 1:
        v = v.mean(axis=0)
    return v / (np.linalg.norm(v) + 1e-9)


def _match(req: str, skills: list[str]) -> bool:
    return any(req == s or (len(req) > 2 and (req in s or s in req)) for s in skills)


def puntuar(req: dict, cand: dict) -> tuple[float, list[str]]:
    """Compara diccionario vs diccionario. Devuelve (score 0-1, habilidades faltantes)."""
    partes, faltan = [], []
    reqs = req.get("habilidades") or []
    if reqs:
        faltan = [r for r in reqs if not _match(r, cand["habilidades"])]
        partes.append((0.7, (len(reqs) - len(faltan)) / len(reqs)))
    if req.get("anios_min"):
        partes.append((0.2, min(1.0, (cand["anios"] or 0) / req["anios_min"])))
    if req.get("idiomas"):
        partes.append((0.1, sum(i in cand["idiomas"] for i in req["idiomas"]) / len(req["idiomas"])))
    if not partes:
        return 0.0, faltan
    return sum(w * v for w, v in partes) / sum(w for w, _ in partes), faltan


# ---------- Base de datos vectorial: Qdrant ----------
# Lección del repo de Azure AI Search: las frases comunes ("Hoja de vida") ensucian el embedding.
# Por eso se limpian y se antepone nombre + habilidades (equivale a title + keywords del índice de Azure).
BOILERPLATE = re.compile(r"\b(hoja de vida|curriculum vitae|curr[ií]culum|cv)\b\s*[:\-]?", re.I)


def texto_para_embedding(nombre: str, p: dict, texto: str) -> str:
    return f"{nombre.replace('_', ' ')}. Habilidades: {', '.join(p['habilidades'])}. {BOILERPLATE.sub('', texto)[:1500]}"


def asegurar_coleccion(dim: int) -> None:
    if qd.collection_exists(COLECCION):
        return
    qd.create_collection(COLECCION, vectors_config=models.VectorParams(size=dim, distance=models.Distance.COSINE))
    if QDRANT_URL:  # en Qdrant Cloud conviene indexar los campos del payload (en modo local no hace falta)
        qd.create_payload_index(COLECCION, "habilidades", models.PayloadSchemaType.KEYWORD)
        qd.create_payload_index(COLECCION, "anios", models.PayloadSchemaType.INTEGER)


def indexar_cv(nombre: str, texto: str) -> dict:
    p = payload_cv(texto)
    vec = embed(texto_para_embedding(nombre, p, texto)).tolist()
    asegurar_coleccion(len(vec))
    qd.upsert(COLECCION, points=[models.PointStruct(
        id=str(uuid.uuid5(uuid.NAMESPACE_DNS, nombre)),   # mismo nombre = se actualiza, no se duplica
        vector=vec,
        payload={"candidato": nombre, **p, "texto": texto})])
    return p


def evidencia(texto: str, req: dict, n: int = 2) -> list[str]:
    frases = [f.strip() for f in re.split(r"(?<=[.!?\n])\s+", texto) if f.strip()]
    terms = req.get("habilidades") or []
    return sorted(frases, key=lambda f: -sum(_palabra(t, norm(f)) for t in terms))[:n]


def buscar_por_payload(req: dict, k: int = 5) -> list[dict]:
    """1) Qdrant filtra por payload (habilidades) y ordena por vector  2) se re-puntúa con el diccionario."""
    if not qd.collection_exists(COLECCION):
        raise ValueError("Primero sube CVs")
    hab = req.get("habilidades") or []
    if not hab:
        raise ValueError("El payload necesita al menos una habilidad en 'habilidades'")
    qvec = embed("Habilidades: " + ", ".join(hab)).tolist()
    filtro = models.Filter(must=[models.FieldCondition(key="habilidades", match=models.MatchAny(any=hab))])
    pts = qd.query_points(COLECCION, query=qvec, query_filter=filtro, limit=k * 5, with_payload=True).points
    if not pts:  # nadie cumple ni una habilidad: caemos a búsqueda puramente semántica
        pts = qd.query_points(COLECCION, query=qvec, limit=k * 5, with_payload=True).points
    res = []
    for pt in pts:
        pl = pt.payload
        score, faltan = puntuar(req, pl)
        res.append({"candidato": pl["candidato"], "score": round(0.75 * score + 0.25 * pt.score, 3),
                    "faltan": faltan,
                    "payload": {c: pl.get(c) for c in ("habilidades", "anios", "idiomas")},
                    "evidencia": evidencia(pl["texto"], req)})
    return sorted(res, key=lambda r: -r["score"])[:k]


def listar_candidatos() -> dict:
    if not qd.collection_exists(COLECCION):
        return {}
    pts, _ = qd.scroll(COLECCION, limit=500, with_payload=["candidato", "habilidades", "anios", "idiomas"])
    return {p.payload["candidato"]: {c: p.payload.get(c) for c in ("habilidades", "anios", "idiomas")} for p in pts}


def borrar_todo() -> None:
    if qd.collection_exists(COLECCION):
        qd.delete_collection(COLECCION)


def agente(pregunta: str, k: int = 3) -> dict:
    req = payload_pregunta(pregunta)                 # 1) pregunta -> payload
    ranking = buscar_por_payload(req, k)             # 2) Qdrant (filtro + vector) + puntuación por payload
    ctx = "\n\n".join(f"[{r['candidato']}] score={r['score']} faltan={r['faltan']}\n"
                      f"payload={r['payload']}\nEvidencia: {' | '.join(r['evidencia'])}" for r in ranking)
    respuesta = llm("Eres un asistente de selección. Usa SOLO los datos dados, justifica el ranking "
                    "con evidencia y menciona lo que falta. No inventes. Español.",
                    f"Pregunta: {pregunta}\nPayload buscado: {req}\n\nTop candidatos:\n{ctx}")  # 3) explicar
    return {"payload": req, "ranking": ranking, "respuesta": respuesta}

## 2. Indexar CVs en Qdrant

In [ ]:
borrar_todo()   # empezar limpio
cvs = {
 "Ana_Torres": "Hoja de vida: Ana Torres. Ingeniera de datos con 5 años de experiencia. Python, SQL, Spark, Airflow y AWS. Pipelines ETL para banca. Inglés avanzado.",
 "Luis_Mendoza": "Hoja de vida: Luis Mendoza. Desarrollador backend, 3 años. FastAPI, Django, PostgreSQL, Docker. APIs REST para una fintech. Conocimientos básicos de machine learning.",
 "Carla_Vega": "Hoja de vida: Carla Vega. Científica de datos, 4 años. PyTorch, NLP, Hugging Face, RAG con LLMs. Chatbot de soporte con embeddings y FAISS. Inglés intermedio.",
 "Pedro_Ruiz": "Hoja de vida: Pedro Ruiz. Diseñador gráfico, 6 años. Photoshop, Illustrator, Figma. Branding y UX/UI. Sin experiencia en programación.",
}
for nombre, texto in cvs.items():
    print(nombre, "->", indexar_cv(nombre, texto))
listar_candidatos()

## 3. Pregunta → payload

In [ ]:
req = payload_pregunta("Busco alguien con Python, RAG y Hugging Face, con más de 3 años de experiencia")
req

## 4. Búsqueda en Qdrant (filtro por payload + vector)

In [ ]:
for r in buscar_por_payload(req, k=4):
    print(f"{r['score']:.3f} | {r['candidato']:<13} | faltan: {r['faltan']}")

Con un payload escrito a mano (solo 1 llamada de embedding, sin LLM):

In [ ]:
buscar_por_payload({"habilidades": ["fastapi", "postgresql"], "anios_min": 2}, k=2)

## 5. Agente completo

In [ ]:
out = agente("¿Quién sabe construir APIs con FastAPI y bases de datos?")
print("Payload:", out["payload"])
print("\n", out["respuesta"])